# PSOD: Basic Tutorial

## Introduction to Pseudo-Supervised Outlier Detection

This notebook provides a comprehensive introduction to PSOD (Pseudo-Supervised Outlier Detection), covering:

1. Installation and setup
2. Basic usage and workflow
3. Understanding outlier scores
4. Visualization techniques
5. Parameter tuning basics

### What is PSOD?

PSOD is an innovative outlier detection method that treats each feature as a target variable and uses prediction errors as outlier scores. This pseudo-supervised approach leverages the power of regression models without requiring labeled data.

## 1. Setup and Installation

First, let's install PSOD and import the necessary libraries.

In [ ]:
# Install PSOD (uncomment if needed)
# !pip install psod

# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# For development - add parent directory to path
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent / 'src'))

# Import PSOD
from psod import PSOD
from psod.visualization import (
    plot_outlier_scores,
    plot_outliers_scatter,
    plot_feature_contributions
)

print("Setup complete!")

## 2. Generate Sample Data

Let's create a synthetic dataset with known outliers to understand how PSOD works.

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Parameters
n_samples = 200
n_outliers = 20
n_features = 5

# Generate normal data (Gaussian distribution)
normal_data = np.random.randn(n_samples - n_outliers, n_features)

# Generate outliers (uniform distribution in wider range)
outliers = np.random.uniform(-5, 5, (n_outliers, n_features))

# Combine data
X = np.vstack([normal_data, outliers])

# Create DataFrame
df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(n_features)])

# Ground truth labels (0 = normal, 1 = outlier)
y_true = np.array([0] * (n_samples - n_outliers) + [1] * n_outliers)

print(f"Dataset shape: {df.shape}")
print(f"Number of outliers: {n_outliers} ({100 * n_outliers / n_samples:.1f}%)")
print(f"\nFirst 5 rows:\n{df.head()}")
print(f"\nData statistics:\n{df.describe()}")

## 3. Basic Outlier Detection with PSOD

Now let's use PSOD to detect outliers in our dataset.

In [ ]:
# Initialize PSOD detector
detector = PSOD(
    min_cols_chosen=0.5,      # Use at least 50% of columns for each regressor
    max_cols_chosen=1.0,      # Use up to 100% of columns
    stdevs_to_outlier=2.0,    # Flag points 2 std deviations from mean
    random_seed=42            # For reproducibility
)

print("PSOD detector initialized with parameters:")
print(f"  min_cols_chosen: {detector.min_cols_chosen}")
print(f"  max_cols_chosen: {detector.max_cols_chosen}")
print(f"  stdevs_to_outlier: {detector.stdevs_to_outlier}")

In [ ]:
# Fit the detector and get outlier scores
outlier_scores = detector.fit_predict(df, return_class=False)

# Get binary outlier labels
outlier_labels = detector.fit_predict(df, return_class=True)

print("Outlier Detection Complete!")
print(f"\nOutlier scores statistics:")
print(f"  Mean: {outlier_scores.mean():.4f}")
print(f"  Std: {outlier_scores.std():.4f}")
print(f"  Min: {outlier_scores.min():.4f}")
print(f"  Max: {outlier_scores.max():.4f}")
print(f"\nDetected {sum(outlier_labels)} outliers")
print(f"True outliers: {sum(y_true)}")

## 4. Evaluate Detection Performance

Let's evaluate how well PSOD detected the outliers.

In [ ]:
from psod import evaluate_outlier_detection

# Evaluate performance
metrics = evaluate_outlier_detection(y_true, outlier_labels, outlier_scores)

print("Performance Metrics:")
print("=" * 40)
print(f"Precision:  {metrics['precision']:.3f}")
print(f"Recall:     {metrics['recall']:.3f}")
print(f"F1-Score:   {metrics['f1']:.3f}")
print(f"ROC-AUC:    {metrics['roc_auc']:.3f}")
print(f"PR-AUC:     {metrics['pr_auc']:.3f}")
print("=" * 40)

# Confusion matrix
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, outlier_labels)
print(f"\nConfusion Matrix:")
print(f"                 Predicted")
print(f"                Normal  Outlier")
print(f"Actual Normal   {cm[0, 0]:6d}  {cm[0, 1]:7d}")
print(f"       Outlier  {cm[1, 0]:6d}  {cm[1, 1]:7d}")

## 5. Visualize Results

Visualization helps us understand the detection results better.

In [ ]:
# Plot outlier scores distribution
fig, ax = plt.subplots(figsize=(12, 6))
plot_outlier_scores(outlier_scores, outlier_labels, ax=ax)
plt.tight_layout()
plt.show()

print(f"\nTop 10 outlier scores:")
top_outliers = np.argsort(outlier_scores)[-10:][::-1]
for idx in top_outliers:
    print(f"  Sample {idx}: score={outlier_scores[idx]:.4f}, true_label={y_true[idx]}")

In [ ]:
# 2D scatter plot (using first two features)
fig, ax = plt.subplots(figsize=(10, 8))
plot_outliers_scatter(
    df[['feature_0', 'feature_1']].values,
    outlier_labels,
    outlier_scores,
    feature_names=['Feature 0', 'Feature 1'],
    ax=ax
)
plt.tight_layout()
plt.show()

## 6. Understanding Feature Contributions

PSOD can help us understand which features contribute most to outlier detection.

In [ ]:
from psod import compute_feature_importance

# Compute feature importance
feature_importance = compute_feature_importance(detector, df)

print("Feature Importance Scores:")
print("=" * 40)
for feature, importance in feature_importance.items():
    print(f"{feature}: {importance:.4f}")
print("=" * 40)

# Visualize feature importance
fig, ax = plt.subplots(figsize=(10, 6))
features = list(feature_importance.keys())
importances = list(feature_importance.values())

ax.barh(features, importances, color='steelblue')
ax.set_xlabel('Importance Score', fontsize=12)
ax.set_title('Feature Importance for Outlier Detection', fontsize=14)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 7. Working with Real Data

Let's apply PSOD to a more realistic scenario with categorical features.

In [ ]:
# Generate more realistic data
np.random.seed(42)
n_samples = 300

# Create a dataset with mixed features
data = {
    'age': np.random.randint(18, 70, n_samples),
    'income': np.random.lognormal(10, 1, n_samples),
    'credit_score': np.random.randint(300, 850, n_samples),
    'num_transactions': np.random.poisson(50, n_samples),
    'category': np.random.choice(['A', 'B', 'C', 'D'], n_samples),
    'region': np.random.choice(['North', 'South', 'East', 'West'], n_samples)
}

df_real = pd.DataFrame(data)

# Add some outliers
n_outliers = 15
outlier_indices = np.random.choice(n_samples, n_outliers, replace=False)

for idx in outlier_indices:
    # Make some extreme values
    df_real.loc[idx, 'income'] *= 10
    df_real.loc[idx, 'num_transactions'] *= 5

print("Realistic dataset created:")
print(f"Shape: {df_real.shape}")
print(f"\nFirst 5 rows:\n{df_real.head()}")
print(f"\nData types:\n{df_real.dtypes}")

In [ ]:
# Initialize PSOD with categorical column specification
detector_real = PSOD(
    cat_columns=['category', 'region'],  # Specify categorical columns
    min_cols_chosen=0.5,
    max_cols_chosen=1.0,
    stdevs_to_outlier=2.5,
    transform_algorithm='yeo-johnson',  # Good for mixed data
    random_seed=42
)

# Detect outliers
scores_real = detector_real.fit_predict(df_real, return_class=False)
labels_real = detector_real.fit_predict(df_real, return_class=True)

print(f"Detected {sum(labels_real)} outliers")
print(f"\nTop 10 outlier samples:")
top_indices = np.argsort(scores_real)[-10:][::-1]
print(df_real.iloc[top_indices])

In [ ]:
# Visualize results
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Outlier scores
plot_outlier_scores(scores_real, labels_real, ax=axes[0])

# Scatter plot of income vs num_transactions
normal_mask = labels_real == 0
outlier_mask = labels_real == 1

axes[1].scatter(
    df_real.loc[normal_mask, 'income'],
    df_real.loc[normal_mask, 'num_transactions'],
    c='blue',
    alpha=0.6,
    s=50,
    label='Normal'
)
axes[1].scatter(
    df_real.loc[outlier_mask, 'income'],
    df_real.loc[outlier_mask, 'num_transactions'],
    c='red',
    alpha=0.8,
    s=100,
    marker='X',
    label='Outlier'
)
axes[1].set_xlabel('Income', fontsize=12)
axes[1].set_ylabel('Number of Transactions', fontsize=12)
axes[1].set_title('Income vs Transactions with Outliers', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Model Persistence

You can save and load trained PSOD models for later use.

In [ ]:
from psod import save_model, load_model

# Save the model
save_model(detector_real, 'psod_model.pkl')
print("Model saved to: psod_model.pkl")

# Load the model
loaded_detector = load_model('psod_model.pkl')
print("Model loaded successfully")

# Verify the loaded model works
scores_loaded = loaded_detector.predict(df_real.head(), return_class=False)
print(f"\nPrediction on first 5 samples:\n{scores_loaded}")

## 9. Summary and Best Practices

### Key Takeaways:

1. **PSOD is easy to use**: Just initialize and call `fit_predict()`
2. **Works with mixed data**: Handles both numerical and categorical features
3. **Interpretable results**: Provides outlier scores and feature importance
4. **Flexible**: Many parameters for tuning to your specific use case

### Best Practices:

1. **Start with default parameters** and tune based on your data
2. **Specify categorical columns** explicitly for better encoding
3. **Use appropriate transformations** for skewed data
4. **Visualize results** to understand what's being detected
5. **Validate with domain knowledge** - not all high scores are true outliers

### Next Steps:

- Explore the **Advanced Tutorial** for more sophisticated techniques
- Check out **Real-World Case Studies** for practical applications
- Experiment with different parameters on your own data

In [ ]:
# Clean up
import os
if os.path.exists('psod_model.pkl'):
    os.remove('psod_model.pkl')
    print("Cleaned up temporary files")